## 1 · Configuration & imports

In [ ]:
import os, sys, subprocess, warnings, json, math, time, zipfile, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# --- install pytorch-forecasting stack if missing (matches submission requirements) ---
def _ensure(pkg, imp=None):
    try:
        __import__(imp or pkg)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

_ensure("pytorch-forecasting==1.8.0", "pytorch_forecasting")
_ensure("lightning==2.6.5", "lightning")

import torch
from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import GroupNormalizer, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.metrics import MAE as PF_MAE

# ------------------------------- paths -------------------------------
DATA_URL   = "https://zenodo.org/records/5849025/files/JapaneseBlackBeefData.zip"
WORK_DIR   = Path(os.environ.get("COW_WORK_DIR", "/kaggle/working" if Path("/kaggle").exists() else "."))
DATA_DIR   = WORK_DIR / "cow_data"          # raw cowN.csv land here
OUTPUT_DIR = WORK_DIR / "cow_outputs"       # predictions, report, checkpoints
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------- resampling / framing ----------------------
BIN         = "1s"      # TODO(tune): resample bin. "1s" → 1 Hz. Coarser = fewer, longer bins.
HORIZON     = 30        # TODO(tune): forecast steps (at BIN="1s" → 30 s ahead)
MIN_ENCODER = 60        # TODO(tune): shortest history window fed to the model
MAX_ENCODER = 120       # TODO(tune): longest history window
STRIDE      = 6         # subsample training windows: keep every STRIDE-th decoder start per series
MIN_SEG_LEN = MIN_ENCODER + HORIZON   # segments shorter than this are dropped

# ------------------------------- TFT ---------------------------------
# Hyper-parameters copied from the winning benchmark config (exp-13 run report).
HIDDEN_SIZE            = 96
HIDDEN_CONTINUOUS_SIZE = 32
ATTENTION_HEADS        = 4
LSTM_LAYERS            = 1
DROPOUT                = 0.15
LEARNING_RATE          = 1e-3
WEIGHT_DECAY           = 1e-4
GRAD_CLIP              = 0.5
BATCH_SIZE             = 64
MAX_EPOCHS             = 60
PATIENCE               = 10
SEEDS                  = [int(s) for s in os.environ.get("COW_SEEDS", "42,7,13").split(",")]  # multi-seed mean; override via COW_SEEDS
ABLATE_LABEL           = os.environ.get("COW_NO_LABEL", "0") == "1"   # ablation: drop the behaviour covariate
TAG                    = os.environ.get("COW_TAG", "14a_cow")         # output filename suffix

# ------------------------- device & smoke test -----------------------
# Also settable from the shell: COW_MPS=1 / COW_SMOKE=1 python ...
USE_MPS          = os.environ.get("COW_MPS", "0") == "1"    # try Apple Metal; CPU is the safe default
LOCAL_SMOKE_TEST = os.environ.get("COW_SMOKE", "0") == "1"  # tiny/fast run to verify the pipeline end-to-end

if torch.cuda.is_available():
    ACCELERATOR, PRECISION = "gpu", "16-mixed"
elif USE_MPS and torch.backends.mps.is_available():
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
    ACCELERATOR, PRECISION = "mps", "32-true"
else:
    ACCELERATOR, PRECISION = "cpu", "32-true"

if LOCAL_SMOKE_TEST:              # fast correctness pass (seconds–minutes on CPU)
    MAX_EPOCHS, SEEDS, MAX_ENCODER = 3, [42], min(MAX_ENCODER, 90)

print(f"torch {torch.__version__} · accelerator={ACCELERATOR} · precision={PRECISION} · work_dir={WORK_DIR}")

## 2 · Download the Zenodo dataset

In [ ]:
import ssl, certifi
zip_path = DATA_DIR / "JapaneseBlackBeefData.zip"

# Kaggle users can also attach the dataset and point COW_RAW_DIR at it to skip the download.
raw_override = os.environ.get("COW_RAW_DIR")
if raw_override:
    DATA_DIR = Path(raw_override)
    print(f"Using pre-attached raw data at {DATA_DIR}")
else:
    if not zip_path.exists():
        print(f"Downloading {DATA_URL} ...")
        # certifi CA bundle: avoids the macOS python.org "CERTIFICATE_VERIFY_FAILED" issue
        ctx = ssl.create_default_context(cafile=certifi.where())
        with urllib.request.urlopen(DATA_URL, context=ctx) as r, open(zip_path, "wb") as fh:
            fh.write(r.read())
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_DIR)

cow_files = sorted(DATA_DIR.rglob("cow*.csv"))
assert cow_files, f"No cowN.csv found under {DATA_DIR}"
print(f"Found {len(cow_files)} cow files:")
for f in cow_files:
    print(" ", f.relative_to(DATA_DIR))

## 3 · Parse, resample, and cast into a panel

Per cow we:
1. read the 25 Hz stream and compute **SMV** = √(AccX²+AccY²+AccZ²) (activity magnitude),
2. resample onto a regular `BIN` grid — `target` = mean SMV in the bin, and the
   **behaviour label** for the bin = its mode (the planned/known covariate),
3. split each cow into **contiguous segments** (gaps in the labelled recording break a
   series), keeping only segments ≥ `MIN_SEG_LEN`. Each `(cow, segment)` becomes one series.


In [ ]:
LABEL_COL = "Label"

def load_cow(path):
    df = pd.read_csv(path)
    # schema: TimeStamp_UNIX (milliseconds), TimeStamp_JST, AccX, AccY, AccZ, Label
    df["timestamp"] = pd.to_datetime(df["TimeStamp_UNIX"], unit="ms", errors="coerce")
    if df["timestamp"].isna().all():                      # fall back to JST string column
        df["timestamp"] = pd.to_datetime(df["TimeStamp_JST"], errors="coerce")
    df["smv"] = np.sqrt(df["AccX"] ** 2 + df["AccY"] ** 2 + df["AccZ"] ** 2)  # 25 Hz signal-vector magnitude
    df[LABEL_COL] = df[LABEL_COL].astype("string").fillna("unlabeled")        # unlabelled stretches -> its own class
    return df[["timestamp", "smv", LABEL_COL]].dropna(subset=["timestamp"])

def resample_cow(df):
    df = df.set_index("timestamp").sort_index()
    grp = df["smv"].resample(BIN)
    count  = grp.count()                                  # samples per bin; 0 marks a true recording gap
    target = grp.std().where(count > 1, 0.0)              # movement intensity (within-bin SMV std)
    target = target.where(count > 0, np.nan)              # keep gaps as NaN so segmentation can split on them
    label  = df[LABEL_COL].resample(BIN).agg(lambda s: s.mode().iat[0] if len(s.mode()) else pd.NA)
    return pd.DataFrame({"target": target, LABEL_COL: label})

def contiguous_segments(res, cow_name):
    """Split a resampled cow into gap-free segments on the BIN grid."""
    present = res["target"].notna().to_numpy()
    segs, start = [], None
    for i, ok in enumerate(present):
        if ok and start is None:
            start = i
        if (not ok or i == len(present) - 1) and start is not None:
            end = i + 1 if (ok and i == len(present) - 1) else i
            block = res.iloc[start:end].copy()
            if len(block) >= MIN_SEG_LEN:
                block["series_id"] = f"{cow_name}_seg{len(segs)}"
                segs.append(block)
            start = None
    return segs

panels = []
for f in cow_files:
    res = resample_cow(load_cow(f))
    res[LABEL_COL] = res[LABEL_COL].ffill().bfill()
    for block in contiguous_segments(res, f.stem):
        panels.append(block)

panel = pd.concat(panels).reset_index().rename(columns={"index": "timestamp"})
panel = panel.sort_values(["series_id", "timestamp"]).reset_index(drop=True)

# per-series contiguous integer time index (0..len-1)
panel["time_idx"] = panel.groupby("series_id").cumcount().astype(int)
panel["series_id"] = panel["series_id"].astype(str)
panel[LABEL_COL]   = panel[LABEL_COL].astype(str)
panel["target"]    = panel["target"].astype(np.float32)

seg_lens = panel.groupby("series_id").size()
print(f"series (cow×segment): {panel['series_id'].nunique()}   rows: {len(panel)}")
print(f"segment length  min/median/max: {seg_lens.min()} / {int(seg_lens.median())} / {seg_lens.max()}")
print(f"behaviour labels: {sorted(panel[LABEL_COL].unique())}")
print(f"target  mean/std: {panel['target'].mean():.3f} / {panel['target'].std():.3f}")

## 4 · Time features & train/validation split

Same calendar-feature construction as the benchmark pipeline (`tft_inference.py`), adapted to
the sub-hourly grid. Validation = the **final `HORIZON` steps of every series**; the model
only ever sees `time_idx < t_train_end` during fitting.

In [ ]:
def add_time_features(df):
    df = df.copy()
    t = pd.to_datetime(df["timestamp"])
    sec_of_min = t.dt.second + t.dt.minute * 60
    df["sec_sin"]  = np.sin(2 * np.pi * sec_of_min / 3600)
    df["sec_cos"]  = np.cos(2 * np.pi * sec_of_min / 3600)
    df["min_sin"]  = np.sin(2 * np.pi * t.dt.minute / 60)
    df["min_cos"]  = np.cos(2 * np.pi * t.dt.minute / 60)
    return df

panel = add_time_features(panel)
TIME_REAL_COLS = ["sec_sin", "sec_cos", "min_sin", "min_cos"]
KNOWN_CATS     = [] if ABLATE_LABEL else [LABEL_COL]   # the planned/known behaviour schedule (dropped in the ablation)
print("ablation (no behaviour covariate):", ABLATE_LABEL)

# split point per series: reserve the last HORIZON steps for validation
seg_lens = panel.groupby("series_id")["time_idx"].transform("max")
panel["is_val"] = panel["time_idx"] > (seg_lens - HORIZON)
t_train_max = int(panel.loc[~panel.is_val, "time_idx"].max()) + 1
n_series    = panel["series_id"].nunique()
print(f"train rows: {(~panel.is_val).sum()}   val rows: {panel.is_val.sum()}   series: {n_series}")

## 5 · Build the `TimeSeriesDataSet`

In [ ]:
known_reals = ["time_idx"] + TIME_REAL_COLS

# time_idx is per-series (each segment starts at 0), so the holdout must be per-series:
# train_ds sees only the training portion (~is_val); the GroupNormalizer/encoders are fit on it.
train_ds = TimeSeriesDataSet(
    panel[~panel.is_val],
    time_idx="time_idx",
    target="target",
    group_ids=["series_id"],
    min_encoder_length=MIN_ENCODER,
    max_encoder_length=MAX_ENCODER,
    min_prediction_length=HORIZON,
    max_prediction_length=HORIZON,
    static_categoricals=["series_id"],
    time_varying_known_categoricals=KNOWN_CATS,
    time_varying_known_reals=known_reals,
    time_varying_unknown_reals=["target"],
    target_normalizer=GroupNormalizer(groups=["series_id"], transformation=None),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
    categorical_encoders={
        "series_id": NaNLabelEncoder(add_nan=True),
        **({} if ABLATE_LABEL else {LABEL_COL: NaNLabelEncoder(add_nan=True)}),
    },
)

# subsample training windows for speed: stride-1 windows at 1 s bins are highly redundant.
if STRIDE and STRIDE > 1:
    dec = train_ds.decoded_index.reset_index(drop=True)          # row-aligned with train_ds.index
    order = dec.sort_values(["series_id", "time_idx_first_prediction"])
    sel = (order.groupby("series_id", sort=False).cumcount() % STRIDE == 0).to_numpy()
    keep = np.zeros(len(dec), dtype=bool)
    keep[order.index[sel]] = True                                # map kept rows back to original positions
    train_ds = train_ds.filter(lambda _idx: keep)

# validation windows: one per series, decoder = the final HORIZON steps (the held-out region),
# encoder = the preceding training steps. predict=True picks exactly that trailing window.
val_ds = TimeSeriesDataSet.from_dataset(train_ds, panel, predict=True, stop_randomization=True)

train_loader = train_ds.to_dataloader(train=True,  batch_size=BATCH_SIZE, num_workers=0)
val_loader   = val_ds.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)
print(f"train windows: {len(train_ds)}   val windows: {len(val_ds)} (expect {n_series})")

## 6 · Train the TFT (multi-seed)

In [ ]:
def train_one(seed):
    seed_everything(seed, workers=True)
    tft = TemporalFusionTransformer.from_dataset(
        train_ds,
        hidden_size=HIDDEN_SIZE,
        hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
        attention_head_size=ATTENTION_HEADS,
        lstm_layers=LSTM_LAYERS,
        dropout=DROPOUT,
        learning_rate=LEARNING_RATE,
        optimizer="adamw",
        weight_decay=WEIGHT_DECAY,
        loss=PF_MAE(),
        log_interval=0,
        reduce_on_plateau_patience=4,
    )
    ckpt_cb = ModelCheckpoint(dirpath=OUTPUT_DIR / "models", filename=f"cow_tft_s{seed}",
                              monitor="val_loss", mode="min", save_top_k=1)
    trainer = Trainer(
        max_epochs=MAX_EPOCHS, accelerator=ACCELERATOR, devices=1, precision=PRECISION,
        gradient_clip_val=GRAD_CLIP, logger=False, enable_progress_bar=True,
        callbacks=[EarlyStopping(monitor="val_loss", patience=PATIENCE, mode="min"), ckpt_cb],
    )
    trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=val_loader)
    best = TemporalFusionTransformer.load_from_checkpoint(ckpt_cb.best_model_path)
    return best

seed_preds = []
for s in SEEDS:
    print(f"\n=== training seed {s} ===")
    model = train_one(s)
    out = model.predict(val_loader, mode="prediction", return_index=True,
                        trainer_kwargs=dict(accelerator=ACCELERATOR, devices=1, logger=False,
                                            enable_progress_bar=False))
    seed_preds.append(out.output.cpu().float().numpy())
    pred_index = out.index

pred = np.mean(seed_preds, axis=0)      # (n_series, HORIZON)
print("prediction matrix:", pred.shape)

## 7 · Evaluate — TFT vs. baselines

In [ ]:
def wape(y, p):  return np.sum(np.abs(y - p)) / np.sum(np.abs(y))
def mae(y, p):   return np.mean(np.abs(y - p))
def rmse(y, p):  return np.sqrt(np.mean((y - p) ** 2))
def mape(y, p):  m = np.abs(y) > 1e-6; return np.mean(np.abs((y[m] - p[m]) / y[m])) * 100
def smape(y, p): d = np.abs(y) + np.abs(p); m = d > 1e-6; return np.mean(2 * np.abs(y[m] - p[m]) / d[m]) * 100

def metric_row(name, y, p):
    return dict(model=name, WAPE=wape(y, p), MAE=mae(y, p), RMSE=rmse(y, p),
                MAPE=mape(y, p), sMAPE=smape(y, p))

# assemble ground-truth val matrix aligned to pred_index ordering
val_true = (panel[panel.is_val]
            .pivot_table(index="series_id", columns=panel[panel.is_val].groupby("series_id").cumcount(),
                         values="target"))
order = [str(s) for s in pred_index["series_id"].tolist()]
y_true = val_true.loc[order].to_numpy(dtype=float)

# --- baselines (required by the assignment: naive last-value + a stronger reference) ---
last_val = (panel[~panel.is_val].sort_values("time_idx").groupby("series_id")["target"].last().loc[order]).to_numpy()
naive_pred  = np.repeat(last_val[:, None], HORIZON, axis=1)                       # naive last-value
smean = (panel[~panel.is_val].groupby("series_id")["target"].mean().loc[order]).to_numpy()
smean_pred  = np.repeat(smean[:, None], HORIZON, axis=1)                          # per-series mean

rows = [
    metric_row("naive_last_value", y_true, naive_pred),
    metric_row("series_mean",      y_true, smean_pred),
    metric_row("TFT (ours)",       y_true, pred),
]
results = pd.DataFrame(rows).set_index("model").round(4)
print(results.to_string())

# save predictions in the leaderboard schema: series_id, timestamp, prediction
val_ts = (panel[panel.is_val].sort_values(["series_id", "time_idx"])
          .groupby("series_id")["timestamp"].apply(list))
out_rows = []
for i, sid in enumerate(order):
    for h in range(HORIZON):
        out_rows.append((sid, val_ts[sid][h], float(pred[i, h])))
pred_df = pd.DataFrame(out_rows, columns=["series_id", "timestamp", "prediction"])
pred_df.to_csv(OUTPUT_DIR / f"validation_predictions_{TAG}.csv", index=False)
results.to_csv(OUTPUT_DIR / f"metrics_{TAG}.csv")
print("saved:", OUTPUT_DIR / f"validation_predictions_{TAG}.csv")

## 8 · Run report

In [ ]:
report = f"""# Run report — {TAG}

Dataset: Japanese Black Beef Cow (Zenodo 5849025), 6 cows, 25 Hz tri-axial accelerometer.
Framing: target = resampled movement intensity (within-bin std of SMV);
known covariate = behaviour label{' — ABLATED (dropped)' if ABLATE_LABEL else ''}.

## Config
- BIN={BIN}  HORIZON={HORIZON}  encoder=[{MIN_ENCODER},{MAX_ENCODER}]  stride={STRIDE}  seeds={SEEDS}
- hidden_size={HIDDEN_SIZE}  heads={ATTENTION_HEADS}  dropout={DROPOUT}  lr={LEARNING_RATE}
- ablate_label={ABLATE_LABEL}
- series (cow×segment): {n_series}   train windows: {len(train_ds)}   val windows: {len(val_ds)}

## Results (validation, lower is better)
{results.to_string()}

## Reading
- TFT must beat `naive_last_value` (the assignment's minimum bar) to count as a transfer success.
- Compare against `series_mean` to show it learned behaviour-conditioned structure, not just a level.
- Compare with-label vs. ABLATE_LABEL=1 to quantify how much the known behaviour covariate contributes.
"""
(OUTPUT_DIR / f"run_report_{TAG}.md").write_text(report)
print(report)